# Séance 8 – TP : simulation d'un moteur de jeu
### Lab : un mini-jeu de donjon en mode texte

| | |
|---|---|
| **Durée estimée** | 85 min (40 + 45 min) |
| **Projet fil rouge** | pas de nouvelle étape (exercice indépendant) – le moteur sera repris en S9 |

**Objectifs**
- Implémenter un diagramme de classes UML complet
- Faire collaborer plusieurs classes (composition)
- Utiliser propriétés et méthodes spéciales dans un vrai programme
- Écrire une boucle de jeu testable

**Mode d'emploi**
- Exécuter une cellule : **Maj + Entrée** ; en cas de comportement étrange : *Redémarrer le noyau et tout exécuter*.
- Complétez les cellules marquées `# TODO`, puis exécutez la cellule **✅ Vérification** qui suit : aucune erreur = réponse correcte.
- Difficulté : ⭐ facile · ⭐⭐ intermédiaire · ⭐⭐⭐ avancé · Indices dans les blocs 💡 repliables.

> ℹ️ Chaque classe utilise les précédentes : **validez chaque exercice avant de passer au suivant**. Les signatures et docstrings sont fournies : complétez les corps marqués `# TODO`.

## Le diagramme de classes du jeu

![UML du jeu](img/s08_game.svg)

**Règles** : dégâts = `max(1, attaque - défense)` · PV entre 0 et le maximum · potion = soin, arme = +attaque, armure = +défense ·
le joueur frappe, l'ennemi riposte s'il survit · ennemi vaincu = XP + butin.

---
## Partie 1 – Les personnages et les objets (40 min)

> **Rappel**
> - Propriété en lecture seule : `@property` sans setter
> - Composition : un attribut contient un autre objet (`self.entity = Entity(...)`)
> - `min()` / `max()` pour borner une valeur

### Exercice 1 ⭐ – La classe Item

Un objet a un nom (`name`), un type (`kind` parmi `"potion"`, `"weapon"`, `"armor"`) et une valeur (`value`).
- `__str__` → `"Potion (+10 PV)"`, `"Épée (+4 ATQ)"`, `"Bouclier (+2 DEF)"` ;
- `__repr__` → `"Item('Potion', 'potion', 10)"`.

In [ ]:
class Item:
    """Objet ramassable : potion (soin), arme (attaque) ou armure (défense)."""

    KINDS = {"potion", "weapon", "armor"}

    def __init__(self, name: str, kind: str, value: int) -> None:
        # TODO
        ...

    def __str__(self) -> str:
        # TODO
        ...

    def __repr__(self) -> str:
        # TODO
        ...

<details><summary>💡 Indice</summary>

Un petit dictionnaire `{"potion": "PV", ...}` évite une cascade de `if`.

</details>

In [ ]:
# ✅ Vérification
assert str(Item("Potion", "potion", 10)) == "Potion (+10 PV)"
assert str(Item("Épée", "weapon", 4)) == "Épée (+4 ATQ)" and str(Item("Bouclier", "armor", 2)) == "Bouclier (+2 DEF)"
assert repr(Item("Potion", "potion", 10)) == "Item('Potion', 'potion', 10)"
assert Item.KINDS == {"potion", "weapon", "armor"}
print("✅ Exercice 1 réussi")

### Exercice 2 ⭐⭐ – La classe Entity

Une entité a un nom, une position `x`, `y`, des PV maximum `max_hp` ; ses PV courants sont stockés dans `_hp` (initialement au maximum).
- propriétés en lecture seule `hp`, `is_alive`, `position` (tuple) ;
- `move(dx, dy)`, `distance_to(other)` (distance de Manhattan : |Δx| + |Δy|) ;
- `take_damage(amount)` et `heal(amount)` bornent les PV et **renvoient** la quantité réellement perdue ou gagnée.

In [ ]:
class Entity:
    """Être vivant positionné sur la carte."""

    def __init__(self, name: str, x: int, y: int, max_hp: int) -> None:
        self.name = name
        self.x = x
        self.y = y
        self.max_hp = max_hp
        self._hp = max_hp

    @property
    def hp(self) -> int:
        # TODO
        ...

    @property
    def is_alive(self) -> bool:
        # TODO
        ...

    @property
    def position(self) -> tuple[int, int]:
        # TODO
        ...

    def move(self, dx: int, dy: int) -> None:
        # TODO
        ...

    def distance_to(self, other: "Entity") -> int:
        """Distance de Manhattan."""
        # TODO
        ...

    def take_damage(self, amount: int) -> int:
        """Retire des PV (sans descendre sous 0) et renvoie les dégâts réellement subis."""
        # TODO
        ...

    def heal(self, amount: int) -> int:
        """Rend des PV (sans dépasser le maximum) et renvoie les PV réellement rendus."""
        # TODO
        ...

    def __str__(self) -> str:
        # TODO
        ...

<details><summary>💡 Indice</summary>

`take_damage` : `lost = min(amount, self._hp)`. `heal` : `gained = min(amount, self.max_hp - self._hp)`.

</details>

In [ ]:
# ✅ Vérification
e = Entity("Test", 2, 3, max_hp=20)
assert e.hp == 20 and e.is_alive and e.position == (2, 3)
e.move(1, -2)
assert e.position == (3, 1) and e.distance_to(Entity("B", 0, 0, 1)) == 4
assert e.take_damage(15) == 15 and e.hp == 5
assert e.heal(100) == 15 and e.hp == 20
assert e.take_damage(50) == 20 and e.hp == 0 and not e.is_alive
assert str(Entity("Aya", 0, 1, 30)) == "Aya (30/30 PV) en (0, 1)"
try:
    e.hp = 10
    raise AssertionError("hp doit être en lecture seule")
except AttributeError:
    pass
print("✅ Exercice 2 réussi")

### Exercice 3 ⭐⭐ – La classe Player

Le joueur **contient** une `Entity` (30 PV) ; il a `attack = 5`, `defense = 2`, `xp = 0` et un inventaire (liste d'`Item`).
- `pick_up(item)` ;
- `use(item_name)` : utilise et **retire** le premier objet de ce nom (insensible à la casse) et renvoie un message :
  potion → soin, arme → `attack += value`, armure → `defense += value` ; message `"Pas de « … » dans l'inventaire"` sinon ;
- `strike(enemy)` : inflige `compute_damage(self.attack, enemy.defense)` à l'ennemi et renvoie les dégâts subis (fonction fournie ci-dessous).

In [ ]:
def compute_damage(attack: int, defense: int) -> int:
    """Règle de combat : au moins 1 point de dégât."""
    return max(1, attack - defense)


class Player:
    """Personnage contrôlé par le joueur."""

    def __init__(self, name: str, x: int = 0, y: int = 0) -> None:
        self.entity = Entity(name, x, y, max_hp=30)
        self.attack = 5
        self.defense = 2
        self.xp = 0
        self.inventory: list[Item] = []

    def pick_up(self, item: Item) -> None:
        # TODO
        ...

    def use(self, item_name: str) -> str:
        """Utilise (et consomme) le premier objet portant ce nom."""
        # TODO
        ...

    def strike(self, enemy: "Enemy") -> int:
        # TODO
        ...

    def __str__(self) -> str:
        return f"{self.entity} · ATQ {self.attack} · DEF {self.defense} · XP {self.xp}"

<details><summary>💡 Indice</summary>

Parcourez l'inventaire ; dès que le nom correspond : `self.inventory.remove(item)`, puis traitez selon `item.kind` et `return` le message.

</details>

In [ ]:
# ✅ Vérification
p = Player("Aya")
assert p.entity.hp == 30 and (p.attack, p.defense, p.xp, p.inventory) == (5, 2, 0, [])
p.pick_up(Item("Épée", "weapon", 4))
p.pick_up(Item("Potion", "potion", 10))
assert p.use("épée") == "Aya s'équipe de Épée : attaque 9" and p.attack == 9
p.entity.take_damage(4)
assert p.use("POTION") == "Aya boit Potion : +4 PV" and p.entity.hp == 30
assert p.inventory == [] and p.use("Potion") == "Pas de « Potion » dans l'inventaire"
p.pick_up(Item("Cotte de mailles", "armor", 3))
p.use("cotte de mailles")
assert p.defense == 5
assert compute_damage(5, 10) == 1
print("✅ Exercice 3 réussi")

### Exercice 4 ⭐⭐ – La classe Enemy et le combat

Un ennemi contient une `Entity` et possède `attack`, `defense`, `xp` (gagnée par le joueur qui le bat) et `loot` (un `Item` ou `None`).
Écrivez `strike(player)`, symétrique de `Player.strike`, puis simulez un duel : le joueur frappe, puis l'ennemi riposte s'il est encore en vie,
jusqu'à la mort de l'un des deux. Comptez les échanges dans `rounds`.

In [ ]:
class Enemy:
    """Ennemi présent sur la carte."""

    def __init__(self, name: str, x: int, y: int, max_hp: int, attack: int, defense: int,
                 xp: int, loot: Item | None = None) -> None:
        self.entity = Entity(name, x, y, max_hp)
        self.attack = attack
        self.defense = defense
        self.xp = xp
        self.loot = loot

    def strike(self, player: Player) -> int:
        # TODO
        ...

    def __str__(self) -> str:
        return f"{self.entity} · ATQ {self.attack} · DEF {self.defense}"


hero = Player("Aya")
goblin = Enemy("Gobelin", 1, 0, max_hp=12, attack=6, defense=1, xp=10)
rounds = 0
# TODO : duel
print(rounds, hero, goblin, sep="\n")

<details><summary>💡 Indice</summary>

Gobelin : 12 PV, le joueur inflige 5 − 1 = 4 par coup → 3 coups ; le gobelin riposte 2 fois : 6 − 2 = 4 dégâts chacune.

</details>

In [ ]:
# ✅ Vérification
assert rounds == 3
assert not goblin.entity.is_alive and hero.entity.hp == 22
troll = Enemy("Troll", 0, 0, max_hp=25, attack=9, defense=3, xp=40)
assert troll.strike(Player("X")) == 7 and Player("Y").strike(troll) == 2
print("✅ Exercice 4 réussi")

---
## Partie 2 – Le monde et la boucle de jeu (45 min)

> **Rappel**
> - Carte : `walls` (ensemble de tuples), `items` (dictionnaire `(x, y) → Item`), `enemies` (liste)
> - `dict.pop(clé, None)` retire et renvoie une valeur (ou `None` si absente)
> - Les méthodes d'action **renvoient une liste de messages** : le moteur est testable sans clavier

### Exercice 5 ⭐⭐ – World : construction et affichage

Complétez la classe `World` pour la construction et les requêtes (les actions `fight`, `move_player` et `play_turn` viendront aux exercices suivants) :
- `add_wall`, `place_item`, `add_enemy` ;
- `in_bounds(x, y)`, `enemy_at(x, y)` (ennemi **vivant** à cette position, ou `None`), `alive_enemies()`, `is_over()` ;
- `render()` : une ligne par `y`, un caractère par `x` avec la priorité `@` > `E` > `#` > `!` > `.`.

La fonction `build_demo_world()` (fournie) construit le donjon de la slide 4.

In [ ]:
DIRECTIONS = {"n": (0, -1), "s": (0, 1), "e": (1, 0), "w": (-1, 0)}


class World:
    """Carte du jeu : murs, objets au sol, ennemis et joueur."""

    def __init__(self, width: int, height: int, player: Player) -> None:
        self.width = width
        self.height = height
        self.player = player
        self.walls: set[tuple[int, int]] = set()
        self.items: dict[tuple[int, int], Item] = {}
        self.enemies: list[Enemy] = []
        self.turn = 0
        self.log: list[str] = []

    # --- Construction de la carte
    def add_wall(self, x: int, y: int) -> None:
        # TODO
        ...

    def place_item(self, item: Item, x: int, y: int) -> None:
        # TODO
        ...

    def add_enemy(self, enemy: Enemy) -> None:
        # TODO
        ...

    # --- Requêtes
    def in_bounds(self, x: int, y: int) -> bool:
        # TODO
        ...

    def enemy_at(self, x: int, y: int) -> Enemy | None:
        # TODO
        ...

    def alive_enemies(self) -> list[Enemy]:
        # TODO
        ...

    def is_over(self) -> bool:
        # TODO
        ...

    def render(self) -> str:
        """Carte en mode texte : @ joueur, E ennemi, ! objet, # mur, . sol."""
        # TODO
        ...

    # --- Actions
    def fight(self, enemy: Enemy) -> list[str]:
        """Un échange de coups : le joueur frappe, puis l'ennemi riposte s'il survit."""
        # TODO
        ...

    def move_player(self, direction: str) -> list[str]:
        """Déplace le joueur ; attaque si un ennemi occupe la case visée."""
        # TODO
        ...

    def play_turn(self, command: str) -> list[str]:
        """Exécute une commande : n/s/e/w, use <objet>, inv, map, stats."""
        # TODO
        ...


def build_demo_world() -> World:
    """Petit donjon de démonstration (7 × 5)."""
    world = World(7, 5, Player("Aya"))
    for x, y in [(2, 0), (2, 1), (2, 3), (4, 1), (4, 2), (4, 3)]:
        world.add_wall(x, y)
    world.place_item(Item("Potion", "potion", 10), 0, 3)
    world.place_item(Item("Épée", "weapon", 4), 1, 4)
    world.place_item(Item("Bouclier", "armor", 2), 3, 0)
    world.add_enemy(Enemy("Gobelin", 3, 2, max_hp=12, attack=6, defense=1, xp=10,
                          loot=Item("Potion", "potion", 10)))
    world.add_enemy(Enemy("Troll", 6, 4, max_hp=25, attack=9, defense=3, xp=40))
    return world


world = build_demo_world()
print(world.render())

<details><summary>💡 Indice</summary>

Dans `render`, deux boucles imbriquées : `for y in range(self.height)` puis `for x in range(self.width)`.

</details>

In [ ]:
# ✅ Vérification
world = build_demo_world()
assert world.render() == "@.#!...\n..#.#..\n...E#..\n!.#.#..\n.!....E"
assert world.in_bounds(6, 4) and not world.in_bounds(7, 0) and not world.in_bounds(0, -1)
assert world.enemy_at(3, 2).entity.name == "Gobelin" and world.enemy_at(0, 0) is None
assert len(world.alive_enemies()) == 2 and not world.is_over()
print("✅ Exercice 5 réussi")

### Exercice 6 ⭐⭐⭐ – Déplacements, ramassage et combats

Si vous avez laissé `fight` et `move_player` en `# TODO` à l'exercice précédent, complétez-les maintenant **dans la cellule de l'exercice 5**, puis réexécutez-la.

- `fight(enemy)` : le joueur frappe ; si l'ennemi survit, il riposte ; sinon le joueur gagne l'XP et le butin. Renvoie les messages.
- `move_player(direction)` : direction inconnue → message ; case hors carte ou mur → `"Impossible d'aller par là"` ;
  ennemi sur la case → `fight` (le joueur ne bouge pas) ; sinon déplacement, puis ramassage de l'objet éventuel.

Cette cellule teste vos méthodes sur le donjon de démonstration.

In [ ]:
world = build_demo_world()
for direction in ["n", "e", "e", "s", "x"]:
    print(direction, world.move_player(direction))

<details><summary>💡 Indice</summary>

`x, y = self.player.entity.x + dx, self.player.entity.y + dy`. Pour l'objet : `item = self.items.pop((x, y), None)`.

</details>

In [ ]:
# ✅ Vérification
w = build_demo_world()
assert w.move_player("n") == ["Impossible d'aller par là"]
assert w.move_player("x") == ["Direction inconnue : x"]
assert w.move_player("e") == ["Aya se déplace en (1, 0)"]
assert w.move_player("e") == ["Impossible d'aller par là"]           # mur en (2, 0)
for d in ["w", "s", "s"]:
    w.move_player(d)
assert w.move_player("s") == ["Aya se déplace en (0, 3)", "Ramassé : Potion (+10 PV)"]
w2 = build_demo_world()
for d in ["s", "s", "e", "e"]:
    w2.move_player(d)
assert w2.player.entity.position == (2, 2)
assert w2.move_player("e") == ["Aya inflige 4 dégâts à Gobelin", "Gobelin riposte : 4 dégâts"]
w2.move_player("e")
assert w2.move_player("e") == ["Aya inflige 4 dégâts à Gobelin", "Gobelin est vaincu ! +10 XP", "Butin : Potion (+10 PV)"]
assert w2.player.xp == 10 and w2.player.entity.position == (2, 2) and len(w2.alive_enemies()) == 1
print("✅ Exercice 6 réussi")

---
### Exercice de synthèse ⭐⭐⭐ – Une partie complète

Complétez `play_turn(command)` (toujours dans la cellule de l'exercice 5) : `n`/`s`/`e`/`w`, `use <objet>`, `inv`, `map`, `stats`.
Chaque commande valide incrémente `turn` ; une commande inconnue renvoie `"Commande inconnue : …"` sans consommer de tour. Les messages sont ajoutés à `log`.

Puis rejouez le scénario `commands` : il doit mener à la **victoire**. Calculez `final_hp`, `turns` et `victory`.

In [ ]:
commands = ["s", "s", "s", "s", "e", "use épée", "inv", "n", "n", "e", "e", "e",
            "e", "n", "n", "use bouclier", "danse", "s", "s", "s", "s", "e", "e",
            "use potion", "e", "e", "e", "e", "e", "stats"]
world = build_demo_world()
# TODO : jouer les commandes tant que la partie n'est pas terminée
final_hp = ...
turns = ...
victory = ...

<details><summary>💡 Indice</summary>

La boucle s'arrête dès que `world.is_over()` est vrai : la commande `stats` finale n'est donc pas jouée.

</details>

In [ ]:
# ✅ Vérification
assert victory is True
assert final_hp == 10 and turns == 28
assert world.player.attack == 9 and world.player.defense == 4 and world.player.xp == 50
assert "Commande inconnue : danse" in world.log
assert [str(i) for i in world.player.inventory] == ["Potion (+10 PV)"]
print("✅ Exercice de synthèse réussi")

---
## Projet fil rouge – pas de nouvelle étape

Le TP est indépendant de GestParc. Le moteur de jeu sera **repris en S9** : `Player` et `Enemy` hériteront d'une classe `Character`, ce qui supprimera la duplication que vous avez remarquée (`attack`, `defense`, `strike`…).

---
## Pour aller plus loin (bonus)

1. **Brouillard de guerre** : `render(radius=2)` n'affiche que les cases à distance ≤ `radius` du joueur (les autres en espace).
2. **Ennemis mobiles** : après chaque tour, chaque ennemi à distance ≤ 3 se rapproche d'une case du joueur (sans traverser les murs).
3. **Niveaux** : tous les 25 XP, le joueur gagne un niveau (+5 PV max, +1 attaque). Utilisez une propriété `level`.
4. **Jouer pour de vrai** : lancez `python scripts/game_engine.py` dans un terminal.

In [ ]:
# À vous